# Lesson 5 — The three-layer caching stack — and the false hit

**Module 2 · ~8 minutes · API key required for the traffic cell**

Lesson 2 was *provider* prompt caching: same prefix, cheaper input. This lesson is the layer that **skips the call entirely**. That 100% saving is why people love semantic cache — and why a hit rate without a false-hit rate is a savings claim with the risk removed from the page.

> **Presenting:** this is the section that makes a risk or compliance person trust the rest of the course.

### What you will be able to do

1. Stack L1 exact, L2 semantic, L3 provider-prefix, L4 model — and know which layers skip the call.
2. Run paraphrased support traffic and read hit rate next to spend.
3. See two queries that are "similar" and must never share a cached answer — then set thresholds by risk class.


### How to work through this notebook

Run cells **top to bottom**. Each section tells you what is about to happen *before* you run the code.

| Marker | What it means |
|---|---|
| **About to happen** | What the next cell will do |
| **Watch for** | The number or field that makes the point — pause on it |
| **Why it matters** | The Monday-morning decision this should change |
| **Presenting:** | Live-demo cue. Students: treat this as the takeaway |

A **cost ledger** prints at the end of every notebook that spends money.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
print(f"\nLive provider: {cfg.provider}")
print("Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.")


  Provider : none (offline)
  Arithmetic cells still run. Live cells will use rehearsal fallbacks.
  Add OPENAI_API_KEY or ANTHROPIC_API_KEY to .env for live calls.
  Rate card: verified 5 Sep 2026 — re-check before presenting.

Live provider: offline
Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.


The cell above loads `.env`, chooses **OpenAI or Anthropic** from the key you have, and prints the three model tiers this notebook will call.

**Watch for:** a banner with `Provider`, `floor`, `mid`, `frontier`.
- If it names a vendor, live cells will spend a few cents.
- If it says `offline`, arithmetic still runs. Live cells print a rehearsal fallback instead of crashing — useful on a plane, not a substitute for a real key on caching / routing / eval lessons.

Switch vendor by setting `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and re-running that cell.


---
## 1. The stack

```
[L1] Exact match     <1ms     byte-identical query        skips the call
  v miss
[L2] Semantic cache  3-10ms   cosine >= threshold         skips the call
  v miss
[L3] Prompt cache    50-200ms identical prefix            90-98% off INPUT only
  v miss
[L4] Model call      0.5-2s   full price
```

L1 and L2 skip the call. L3 only discounts the input. They are complements, not alternatives.

**About to happen.** We build a tiny in-memory stack with a toy bag-of-words embedding. In production you swap in a real embedding model — the **mechanics and the failure modes stay the same**; only the threshold changes.

**Watch for:** `threshold=0.85`. We will use that number against two queries that should never collide.


In [2]:
import hashlib, math
from collections import Counter

STOP = {"the","is","a","an","what","how","do","does","did","can","could","you","your",
        "me","my","i","about","tell","please","of","to","for","and","it","in","on","are"}

def embed(text):
    """Toy bag-of-words embedding. Swap for a real embedding model in production —
    the cache mechanics and the failure modes below are identical; only the threshold changes."""
    words = []
    for w in text.lower().replace("?", "").replace(".", "").replace(",", "").split():
        w = w.rstrip("s") if len(w) > 4 and w.endswith("s") else w
        if w not in STOP and len(w) > 2:
            words.append(w)
    return Counter(words)

def cos(a, b):
    common = set(a) & set(b)
    num = sum(a[w] * b[w] for w in common)
    den = math.sqrt(sum(v * v for v in a.values())) * math.sqrt(sum(v * v for v in b.values()))
    return num / den if den else 0.0

class CacheStack:
    def __init__(self, threshold=0.85):
        self.exact, self.semantic, self.threshold = {}, [], threshold
        self.stats = Counter()

    def get(self, q):
        key = hashlib.sha256(q.encode()).hexdigest()
        if key in self.exact:
            self.stats["L1_exact"] += 1
            return self.exact[key], "L1_exact", 1.0
        qv = embed(q)
        best, best_s = None, 0.0
        for cv, cq, ans in self.semantic:
            s = cos(qv, cv)
            if s > best_s:
                best, best_s = (cq, ans), s
        if best and best_s >= self.threshold:
            self.stats["L2_semantic"] += 1
            return best[1], "L2_semantic", best_s
        self.stats["L4_miss"] += 1
        return None, "L4_miss", best_s

    def put(self, q, ans):
        self.exact[hashlib.sha256(q.encode()).hexdigest()] = ans
        self.semantic.append((embed(q), q, ans))

cache = CacheStack(threshold=0.85)
print("cache ready  threshold=", cache.threshold)


cache ready  threshold= 0.85


---
## 2. Run realistic traffic — with the paraphrases every support queue has

**About to happen.** Ten support questions: exact repeats, paraphrases, and a couple of genuinely new ones. On a miss we call the model and store the answer. On a hit we skip the call.

**Watch for:** lines tagged `L1_exact` or `L2_semantic` at `$0.000000`, and the final hit rate. An 70–80% hit rate on FAQ-like traffic is normal. That is the good news. The next cell is the bad news.


In [3]:
SYSTEM = "You are a concise support assistant for Northwind Logistics. Answer in one sentence."

TRAFFIC = [
    "What is your refund policy?",
    "What is your refund policy?",                 # exact repeat
    "Can you tell me about the refund policy?",    # paraphrase
    "How do refunds work here?",                   # paraphrase
    "How do I track my shipment?",
    "How can I track my shipment?",                # paraphrase
    "What are your delivery hours?",
    "What is your refund policy?",                 # exact repeat again
    "Tell me how to track a shipment",             # paraphrase
    "Do you ship internationally?",
]

spent = 0.0
for q in TRAFFIC:
    ans, layer, score = cache.get(q)
    if ans is None:
        r = complete(q, system=SYSTEM, model=MODELS.mid, max_tokens=90,
                     label=f"MISS  {q[:34]}")
        ans = r.text
        spent += r.usd
        cache.put(q, ans)
    else:
        print(f"{layer:<12} {q[:34]:<36} sim={score:.2f}   $0.000000  (call skipped)")

hits = cache.stats["L1_exact"] + cache.stats["L2_semantic"]
print(f"\nhit rate      {hits}/{len(TRAFFIC)} = {hits / len(TRAFFIC):.0%}")
print(f"spent         {usd(spent)}")
print(f"saving        {1 - cache.stats['L4_miss'] / len(TRAFFIC):.0%} of all calls eliminated")


⚠ No API key — using a rehearsal result.
MISS  What is your refund policy?             $0.000842   in=21      out=80     cw=0       cr=0       offline fallback
L1_exact     What is your refund policy?          sim=1.00   $0.000000  (call skipped)
L2_semantic  Can you tell me about the refund p   sim=1.00   $0.000000  (call skipped)
⚠ No API key — using a rehearsal result.
MISS  How do refunds work here?               $0.000842   in=21      out=80     cw=0       cr=0       offline fallback
⚠ No API key — using a rehearsal result.
MISS  How do I track my shipment?             $0.000844   in=22      out=80     cw=0       cr=0       offline fallback
L2_semantic  How can I track my shipment?         sim=1.00   $0.000000  (call skipped)
⚠ No API key — using a rehearsal result.
MISS  What are your delivery hours?           $0.000842   in=21      out=80     cw=0       cr=0       offline fallback
L1_exact     What is your refund policy?          sim=1.00   $0.000000  (call skipped)
L2_semantic 

---
## 3. THE FALSE HIT — the demo nobody runs

**About to happen.** Two queries, high cosine similarity, **opposite** correct answers. One cancels a shipment. The other cancels a recurring contract. One of those actions is hard to unwind.

**Watch for:** `would be served from cache: True` (or a similarity that clears 0.85). The cache has no concept of what the action costs.

**Why it matters.** You can drive hit rate to 90% by lowering the threshold. That is not a win. Semantic cache is situational: FAQ-like, informational traffic — not transactional, legal, medical, or tenant-specific answers.

> **Presenting:** let the two sentences sit on screen for a beat before you talk.


In [4]:
q_a = "Please cancel the order."
q_b = "Please cancel the standing order."

sim = cos(embed(q_a), embed(q_b))
print(f"cosine similarity: {sim:.3f}")
print(f"threshold:         {cache.threshold}")
print(f"-> would be served from cache: {sim >= cache.threshold}")
print()
print("One cancels a single shipment. The other cancels a recurring contract.")
print("The cache cannot tell. It has no concept of what the action costs.")


cosine similarity: 0.816
threshold:         0.85
-> would be served from cache: False

One cancels a single shipment. The other cancels a recurring contract.
The cache cannot tell. It has no concept of what the action costs.


### The controls — threshold by risk class, never globally

**About to happen.** A four-row policy: informational can be cached loosely; account data needs a high threshold and a short TTL; transactional and legal/medical are **not cacheable**.

**Watch for:** `cacheable=False` on the last two rows. Also namespace per tenant, per user, and per entitlement. A cache that crosses a permission boundary is a data incident, not a cost win.


In [5]:
RISK_TIERS = {
    "informational": dict(threshold=0.82, cacheable=True,  ttl_s=86400),
    "account":       dict(threshold=0.95, cacheable=True,  ttl_s=300),
    "transactional": dict(threshold=1.00, cacheable=False, ttl_s=0),
    "legal_medical": dict(threshold=1.00, cacheable=False, ttl_s=0),
}
for tier, cfg_t in RISK_TIERS.items():
    print(f"{tier:<16} threshold={cfg_t['threshold']:<6} "
          f"cacheable={str(cfg_t['cacheable']):<6} ttl={cfg_t['ttl_s']}s")
print()
print("Plus: namespace the cache per tenant, per user, and per entitlement level.")
print("A cache that crosses a permission boundary is a data incident, not a cost win.")


informational    threshold=0.82   cacheable=True   ttl=86400s
account          threshold=0.95   cacheable=True   ttl=300s
transactional    threshold=1.0    cacheable=False  ttl=0s
legal_medical    threshold=1.0    cacheable=False  ttl=0s

Plus: namespace the cache per tenant, per user, and per entitlement level.
A cache that crosses a permission boundary is a data incident, not a cost win.


### Measure the pair, always

**About to happen.** We print hit rate **and** false-hit rate on the same report. The false-hit count here is 0 because we have not sampled humans — that is honest. In production you review a sample every week.

**Watch for:** the warning threshold at 2% false hits. Above that, raise the threshold; you are buying savings with wrong answers.


In [6]:
def report(hits, total, false_hits, sampled):
    print(f"hit rate        {hits / total:.1%}")
    print(f"false-hit rate  {false_hits / max(1, sampled):.1%}  (from {sampled} human-reviewed samples)")
    print(f"net saving      {(hits - false_hits) / total:.1%}")
    if false_hits / max(1, sampled) > 0.02:
        print("\nWARNING: raise the threshold. You are buying savings with wrong answers.")

report(hits=cache.stats["L1_exact"] + cache.stats["L2_semantic"],
       total=len(TRAFFIC), false_hits=0, sampled=20)
print()
print("You can drive hit rate to 90% by lowering the threshold. That is not a win.")
print("Semantic cache is situational: FAQ-like traffic, not transactional / legal / medical.")


hit rate        50.0%
false-hit rate  0.0%  (from 20 human-reviewed samples)
net saving      50.0%

You can drive hit rate to 90% by lowering the threshold. That is not a win.
Semantic cache is situational: FAQ-like traffic, not transactional / legal / medical.


In [7]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.004210


,label,model,input,output,cache_write,cache_read,usd,note
0,MISS What is your refund policy?,claude-sonnet-5,21,80,0,0,0.000842,offline fallback
1,MISS How do refunds work here?,claude-sonnet-5,21,80,0,0,0.000842,offline fallback
2,MISS How do I track my shipment?,claude-sonnet-5,22,80,0,0,0.000844,offline fallback
3,MISS What are your delivery hours?,claude-sonnet-5,21,80,0,0,0.000842,offline fallback
4,MISS Do you ship internationally?,claude-sonnet-5,20,80,0,0,0.000840,offline fallback


---
## Takeaways

- L1 and L2 **skip the call**. L3 only discounts the input. Stack them; do not pick one.
- Set the threshold **by risk class**, not globally.
- Namespace by tenant, user and entitlement. TTL by how fast the underlying truth changes.
- **Report hit rate and false-hit rate on the same dashboard, always.**

**Try on Monday:** pick one FAQ-like flow, add an exact-match cache, and log cosine similarity on the misses. Do **not** enable semantic serving on anything that books, cancels, pays, or cites a regulation until you have a false-hit sample.
